# Code Analysis Deep-Dive

Explore the HIVE-AGENT analysis modules interactively:
- AST + Tree-sitter parsing
- Dependency graph construction
- Cyclomatic complexity
- Code smell detection
- Security scanning
- Dead code identification
- Similarity engine

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json, textwrap
import plotly.graph_objects as go
import networkx as nx

## Sample Codebase

In [ ]:
FILES = {
    'auth.py': '''
import hashlib, os

SECRET = "hardcoded_secret_123"  # noqa: S105

def hash_password(password: str) -> str:
    salt = os.urandom(16)
    return hashlib.sha256(salt + password.encode()).hexdigest()

def check_password(stored: str, provided: str) -> bool:
    return stored == hash_password(provided)
''',
    'utils.py': '''
from auth import hash_password

def create_user(username: str, password: str) -> dict:
    return {"username": username, "password_hash": hash_password(password)}

def _unused_helper():
    pass
'''
}
print('Sample files ready:', list(FILES.keys()))

## Dependency Graph

In [ ]:
from analysis.dependency_graph import DependencyGraph

dg = DependencyGraph()
graph = dg.build(FILES)
print('Nodes:', list(graph.nodes()))
print('Edges:', list(graph.edges()))

In [ ]:
# Visualise dependency graph with Plotly
pos = nx.spring_layout(graph, seed=42)
edge_x, edge_y = [], []
for u, v in graph.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

node_x = [pos[n][0] for n in graph.nodes()]
node_y = [pos[n][1] for n in graph.nodes()]

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
                         line=dict(width=1, color='#888')))
fig.add_trace(go.Scatter(x=node_x, y=node_y, mode='markers+text',
                         text=list(graph.nodes()), textposition='top center',
                         marker=dict(size=12, color='#00d4ff')))
fig.update_layout(title='Dependency Graph', showlegend=False,
                  xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
fig.show()

## Complexity Scores

In [ ]:
from analysis.complexity_scorer import ComplexityScorer

scorer = ComplexityScorer()
for fname, code in FILES.items():
    scores = scorer.score(code)
    print(f'{fname}: {scores}')

## Code Smell Detection

In [ ]:
from analysis.code_smell_detector import CodeSmellDetector

detector = CodeSmellDetector()
for fname, code in FILES.items():
    smells = detector.detect(code)
    print(f'{fname}: {smells}')

## Security Scan

In [ ]:
from analysis.security_scanner import SecurityScanner

scanner = SecurityScanner()
for fname, code in FILES.items():
    issues = scanner.scan(code, filename=fname)
    print(f'{fname}: {issues}')

## Dead Code Detection

In [ ]:
from analysis.dead_code_finder import DeadCodeFinder

finder = DeadCodeFinder()
dead = finder.find(FILES)
print('Dead code candidates:', dead)

## Similarity Engine

In [ ]:
from analysis.similarity_engine import SimilarityEngine

engine = SimilarityEngine()
snippets = [
    'def add(a, b): return a + b',
    'def sum_two(x, y): return x + y',
    'def multiply(a, b): return a * b',
]
pairs = engine.find_similar_pairs(snippets, threshold=0.5)
print('Similar pairs:', pairs)